In [ ]:
# ============================================================
# BIASCOPE
# Notebook 03 — Baseline Models & Parameter-Efficient Fine-Tuning
# ============================================================
#
# Objective:
#
# 1. Load processed MBIB data
# 2. Establish TF-IDF + Logistic Regression baseline
# 3. Fine-tune RoBERTa using full parameter training
# 4. Fine-tune RoBERTa using LoRA
# 5. Compare:
#       - Accuracy
#       - Precision
#       - Recall
#       - F1
#       - ROC-AUC
#       - Trainable parameters
#       - Training efficiency
#
# Models:
#
#   Baseline:
#       TF-IDF + Logistic Regression
#
#   Deep Learning:
#       RoBERTa-base Full Fine-Tuning
#
#   PEFT:
#       RoBERTa-base + LoRA
#
# ============================================================

PROJECT_NAME = "BIASCOPE"

print(PROJECT_NAME)
print("Notebook 03: Baseline Models & LoRA")

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from datasets import Dataset

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# ============================================================
# BIASCOPE DIRECTORIES
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "outputs"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# DATASET PATHS
# ============================================================

TRAIN_PATH = PROCESSED_DIR / "mbib_train.csv"
VAL_PATH = PROCESSED_DIR / "mbib_validation.csv"
TEST_PATH = PROCESSED_DIR / "mbib_test.csv"

print("TRAIN:", TRAIN_PATH)
print("VAL:", VAL_PATH)
print("TEST:", TEST_PATH)

print("\nFiles exist?")

print("Train:", TRAIN_PATH.exists())
print("Val:", VAL_PATH.exists())
print("Test:", TEST_PATH.exists())

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
display(train_df.head())

print("\nColumns:")
print(train_df.columns.tolist())

print("\nTrain shape:")
print(train_df.shape)

print("\nLabel distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nDataset ID distribution:")
print(train_df["dataset_id"].value_counts().sort_index())

In [ ]:
X_train = train_df["text"].astype(str)
y_train = train_df["label"]

X_val = val_df["text"].astype(str)
y_val = val_df["label"]

X_test = test_df["text"].astype(str)
y_test = test_df["label"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=100000
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_val_tfidf = tfidf.transform(X_val)

X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

print("\nVocabulary size:", len(tfidf.vocabulary_))

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=SEED
)

start_time = time.time()

lr_model.fit(
    X_train_tfidf,
    y_train
)

lr_training_time = time.time() - start_time

print(f"Training time: {lr_training_time:.2f} seconds")

In [ ]:
def evaluate_model(
    model_name,
    y_true,
    y_pred,
    y_prob=None
):

    results = {
        "model": model_name,

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        )
    }

    if y_prob is not None:

        results["roc_auc"] = roc_auc_score(
            y_true,
            y_prob
        )

    else:

        results["roc_auc"] = np.nan

    return results

In [ ]:
start_time = time.time()

lr_pred = lr_model.predict(
    X_test_tfidf
)

lr_prob = lr_model.predict_proba(
    X_test_tfidf
)[:, 1]

lr_inference_time = time.time() - start_time

lr_results = evaluate_model(
    "TF-IDF + Logistic Regression",
    y_test,
    lr_pred,
    lr_prob
)

lr_results["training_time_sec"] = lr_training_time
lr_results["inference_time_sec"] = lr_inference_time

print(classification_report(
    y_test,
    lr_pred,
    target_names=["Neutral", "Biased"]
))

print("\nResults:")
print(lr_results)

In [ ]:
cm = confusion_matrix(
    y_test,
    lr_pred
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Neutral", "Biased"],
    yticklabels=["Neutral", "Biased"]
)

plt.title("TF-IDF + Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
# ============================================================
# RoBERTa Configuration
# ============================================================

MODEL_NAME = "roberta-base"

MAX_LENGTH = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", NUM_EPOCHS)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
train_hf = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

val_hf = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

test_hf = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print(train_hf)
print(val_hf)
print(test_hf)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_hf.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing training data"
)

val_tokenized = val_hf.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing validation data"
)

test_tokenized = test_hf.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing test data"
)

print(train_tokenized)

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator ready.")

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    probabilities = torch.softmax(
        torch.tensor(logits),
        dim=-1
    ).numpy()[:, 1]

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),

        "precision": precision_score(
            labels,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            labels,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            labels,
            predictions,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            labels,
            probabilities
        )
    }

print("Metrics function ready.")

In [ ]:
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

total_params = sum(
    p.numel()
    for p in roberta_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in roberta_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.2f}%"
)

In [ ]:
full_ft_args = TrainingArguments(
    output_dir=str(
        OUTPUT_DIR / "roberta_full"
    ),

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=100,

    save_total_limit=1,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=SEED
)

print("Training arguments ready.")

In [ ]:
full_trainer = Trainer(
    model=roberta_model,

    args=full_ft_args,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("RoBERTa Full Fine-Tuning Trainer ready.")

In [ ]:
start_time = time.time()

full_train_output = full_trainer.train()

full_training_time = time.time() - start_time

print(
    f"\nFull fine-tuning completed "
    f"in {full_training_time / 60:.2f} minutes."
)

In [ ]:
full_test_output = full_trainer.predict(
    test_tokenized
)

print(full_test_output.metrics)

In [ ]:
full_logits = full_test_output.predictions

full_pred = np.argmax(
    full_logits,
    axis=-1
)

full_prob = torch.softmax(
    torch.tensor(full_logits),
    dim=-1
).numpy()[:, 1]

full_results = evaluate_model(
    "RoBERTa Full Fine-Tuning",
    y_test,
    full_pred,
    full_prob
)

full_results["training_time_sec"] = full_training_time

print(
    classification_report(
        y_test,
        full_pred,
        target_names=["Neutral", "Biased"]
    )
)

print("\nRoBERTa Full FT Results:")
print(full_results)

In [ ]:
cm = confusion_matrix(
    y_test,
    full_pred
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Neutral", "Biased"],
    yticklabels=["Neutral", "Biased"]
)

plt.title("RoBERTa Full Fine-Tuning")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
# ============================================================
# RoBERTa + LoRA
# ============================================================

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

print("LoRA configuration:")
print("Rank (r):", LORA_R)
print("Alpha:", LORA_ALPHA)
print("Dropout:", LORA_DROPOUT)

In [ ]:
lora_base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("Fresh RoBERTa model loaded.")

In [ ]:
!pip uninstall -y torchao

In [ ]:
import importlib.metadata

try:
    version = importlib.metadata.version("torchao")
    print("torchao version:", version)
except importlib.metadata.PackageNotFoundError:
    print("✅ torchao is not installed")

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,

    r=LORA_R,

    lora_alpha=LORA_ALPHA,

    lora_dropout=LORA_DROPOUT,

    target_modules=[
        "query",
        "value"
    ],

    bias="none"
)

lora_model = get_peft_model(
    lora_base_model,
    lora_config
)

print("LoRA successfully applied.")

In [ ]:
total_lora_params = sum(
    p.numel()
    for p in lora_model.parameters()
)

trainable_lora_params = sum(
    p.numel()
    for p in lora_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_lora_params:,}")
print(f"Trainable parameters: {trainable_lora_params:,}")

print(
    f"Trainable percentage: "
    f"{100 * trainable_lora_params / total_lora_params:.4f}%"
)

print("\nFull Fine-Tuning:")
print(f"Trainable parameters: {trainable_params:,}")

print("\nLoRA:")
print(f"Trainable parameters: {trainable_lora_params:,}")

print(
    f"\nParameter reduction: "
    f"{100 * (1 - trainable_lora_params / trainable_params):.2f}%"
)

In [ ]:
lora_args = TrainingArguments(
    output_dir=str(
        OUTPUT_DIR / "roberta_lora"
    ),

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    logging_steps=100,

    save_total_limit=1,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=SEED
)

print("LoRA training arguments ready.")

In [ ]:
lora_trainer = Trainer(
    model=lora_model,

    args=lora_args,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("LoRA Trainer ready.")

In [ ]:
start_time = time.time()

lora_train_output = lora_trainer.train()

lora_training_time = time.time() - start_time

print(
    f"\nLoRA training completed "
    f"in {lora_training_time / 60:.2f} minutes."
)

In [ ]:
lora_test_output = lora_trainer.predict(
    test_tokenized
)

print(lora_test_output.metrics)

In [ ]:
lora_logits = lora_test_output.predictions

lora_pred = np.argmax(
    lora_logits,
    axis=-1
)

lora_prob = torch.softmax(
    torch.tensor(lora_logits),
    dim=-1
).numpy()[:, 1]

lora_results = evaluate_model(
    "RoBERTa + LoRA",
    y_test,
    lora_pred,
    lora_prob
)

lora_results["training_time_sec"] = lora_training_time

print(
    classification_report(
        y_test,
        lora_pred,
        target_names=["Neutral", "Biased"]
    )
)

print("\nRoBERTa + LoRA Results:")
print(lora_results)

In [ ]:
cm = confusion_matrix(
    y_test,
    lora_pred
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Neutral", "Biased"],
    yticklabels=["Neutral", "Biased"]
)

plt.title("RoBERTa + LoRA")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
# ============================================================
# Improved LoRA Experiment
# ============================================================

IMPROVED_LORA_R = 16
IMPROVED_LORA_ALPHA = 32
IMPROVED_LORA_DROPOUT = 0.1

IMPROVED_LR = 1e-4

print("Improved LoRA configuration")
print("Rank:", IMPROVED_LORA_R)
print("Alpha:", IMPROVED_LORA_ALPHA)
print("Dropout:", IMPROVED_LORA_DROPOUT)
print("Learning rate:", IMPROVED_LR)

In [ ]:
improved_lora_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("Fresh RoBERTa model loaded.")

In [ ]:
improved_lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,

    r=IMPROVED_LORA_R,

    lora_alpha=IMPROVED_LORA_ALPHA,

    lora_dropout=IMPROVED_LORA_DROPOUT,

    target_modules=[
        "query",
        "key",
        "value",
        "dense"
    ],

    bias="none"
)

improved_lora_model = get_peft_model(
    improved_lora_base,
    improved_lora_config
)

print("Improved LoRA successfully applied.")

In [ ]:
improved_total_params = sum(
    p.numel()
    for p in improved_lora_model.parameters()
)

improved_trainable_params = sum(
    p.numel()
    for p in improved_lora_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {improved_total_params:,}")
print(f"Trainable parameters: {improved_trainable_params:,}")

print(
    f"Trainable percentage: "
    f"{100 * improved_trainable_params / improved_total_params:.4f}%"
)

print(
    f"\nReduction vs Full FT: "
    f"{100 * (1 - improved_trainable_params / trainable_params):.2f}%"
)

In [ ]:
improved_lora_args = TrainingArguments(
    output_dir=str(
        OUTPUT_DIR / "roberta_lora_improved"
    ),

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=IMPROVED_LR,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=100,

    save_total_limit=1,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=SEED
)

print("Improved LoRA training arguments ready.")

In [ ]:
improved_lora_trainer = Trainer(
    model=improved_lora_model,

    args=improved_lora_args,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("Improved LoRA Trainer ready.")

In [ ]:
start_time = time.time()

improved_lora_train_output = improved_lora_trainer.train()

improved_lora_training_time = time.time() - start_time

print(
    f"\nImproved LoRA training completed "
    f"in {improved_lora_training_time / 60:.2f} minutes."
)

In [ ]:
improved_lora_test_output = improved_lora_trainer.predict(
    test_tokenized
)

print(improved_lora_test_output.metrics)

In [ ]:
improved_logits = improved_lora_test_output.predictions

improved_pred = np.argmax(
    improved_logits,
    axis=-1
)

improved_prob = torch.softmax(
    torch.tensor(improved_logits),
    dim=-1
).numpy()[:, 1]

improved_lora_results = evaluate_model(
    "RoBERTa + Improved LoRA",
    y_test,
    improved_pred,
    improved_prob
)

improved_lora_results["training_time_sec"] = (
    improved_lora_training_time
)

print(
    classification_report(
        y_test,
        improved_pred,
        target_names=["Neutral", "Biased"]
    )
)

print("\nImproved LoRA Results:")
print(improved_lora_results)

In [ ]:
# ============================================================
# Final Model Comparison
# ============================================================

results_df = pd.DataFrame([
    lr_results,
    full_results,
    lora_results,
    improved_lora_results
])

results_df

In [ ]:
comparison_df = results_df[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "training_time_sec"
    ]
].copy()

comparison_df["accuracy"] *= 100
comparison_df["precision"] *= 100
comparison_df["recall"] *= 100
comparison_df["f1"] *= 100
comparison_df["roc_auc"] *= 100

comparison_df = comparison_df.round(2)

comparison_df

In [ ]:
plt.figure(figsize=(10, 6))

plot_df = comparison_df.copy()

x = np.arange(len(plot_df))
width = 0.2

plt.bar(
    x - 1.5 * width,
    plot_df["accuracy"],
    width,
    label="Accuracy"
)

plt.bar(
    x - 0.5 * width,
    plot_df["precision"],
    width,
    label="Precision"
)

plt.bar(
    x + 0.5 * width,
    plot_df["recall"],
    width,
    label="Recall"
)

plt.bar(
    x + 1.5 * width,
    plot_df["f1"],
    width,
    label="F1"
)

plt.xticks(
    x,
    [
        "TF-IDF + LR",
        "RoBERTa Full FT",
        "LoRA v1",
        "Improved LoRA"
    ],
    rotation=15
)

plt.ylabel("Score (%)")
plt.title("BIASCOPE Model Performance Comparison")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Parameter Efficiency
# ============================================================

parameter_df = pd.DataFrame({
    "model": [
        "RoBERTa Full FT",
        "RoBERTa LoRA v1",
        "RoBERTa Improved LoRA"
    ],

    "total_parameters": [
        total_params,
        total_lora_params,
        improved_total_params
    ],

    "trainable_parameters": [
        trainable_params,
        trainable_lora_params,
        improved_trainable_params
    ]
})

parameter_df["trainable_percentage"] = (
    parameter_df["trainable_parameters"]
    / parameter_df["total_parameters"]
    * 100
)

parameter_df

In [ ]:
for _, row in parameter_df.iterrows():

    print(f"\n{row['model']}")

    print(
        f"Total parameters: "
        f"{row['total_parameters']:,}"
    )

    print(
        f"Trainable parameters: "
        f"{row['trainable_parameters']:,}"
    )

    print(
        f"Trainable percentage: "
        f"{row['trainable_percentage']:.4f}%"
    )

In [ ]:
best_pred = full_pred

cm = confusion_matrix(
    y_test,
    best_pred
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Neutral", "Biased"],
    yticklabels=["Neutral", "Biased"]
)

plt.title("Best Model — RoBERTa Full Fine-Tuning")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Error Analysis
# ============================================================

error_df = test_df.copy()

error_df["true_label"] = y_test.values
error_df["predicted_label"] = full_pred

error_df["correct"] = (
    error_df["true_label"]
    == error_df["predicted_label"]
)

error_df["prediction_confidence"] = np.max(
    torch.softmax(
        torch.tensor(full_logits),
        dim=-1
    ).numpy(),
    axis=1
)

errors_only = error_df[
    ~error_df["correct"]
].copy()

print(
    "Total test samples:",
    len(error_df)
)

print(
    "Incorrect predictions:",
    len(errors_only)
)

print(
    "Error rate:",
    len(errors_only) / len(error_df)
)

In [ ]:
display(
    errors_only[
        [
            "text",
            "true_label",
            "predicted_label",
            "prediction_confidence"
        ]
    ].head(20)
)

In [ ]:
false_positives = error_df[
    (error_df["true_label"] == 0) &
    (error_df["predicted_label"] == 1)
]

false_negatives = error_df[
    (error_df["true_label"] == 1) &
    (error_df["predicted_label"] == 0)
]

print("False Positives (Neutral → Biased):", len(false_positives))
print("False Negatives (Biased → Neutral):", len(false_negatives))

In [ ]:
RESULTS_DIR = OUTPUT_DIR / "model_comparison"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

comparison_df.to_csv(
    RESULTS_DIR / "model_comparison.csv",
    index=False
)

parameter_df.to_csv(
    RESULTS_DIR / "parameter_efficiency.csv",
    index=False
)

errors_only.to_csv(
    RESULTS_DIR / "roberta_full_errors.csv",
    index=False
)

print("Results saved successfully.")

print(
    RESULTS_DIR
)

In [ ]:
BEST_MODEL_DIR = MODEL_DIR / "roberta_full_best"

full_trainer.save_model(
    str(BEST_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(BEST_MODEL_DIR)
)

print(
    "RoBERTa Full FT model saved to:"
)

print(BEST_MODEL_DIR)

In [ ]:
LORA_MODEL_DIR = MODEL_DIR / "roberta_improved_lora"

improved_lora_trainer.save_model(
    str(LORA_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(LORA_MODEL_DIR)
)

print(
    "Improved LoRA model saved to:"
)

print(LORA_MODEL_DIR)

In [ ]:
experiment_config = {
    "project": "BIASCOPE",

    "dataset": "MBIB",

    "task": "Binary News Bias Detection",

    "labels": {
        "0": "Neutral",
        "1": "Biased"
    },

    "models": {

        "tfidf_lr": {
            "type": "TF-IDF + Logistic Regression"
        },

        "roberta_full": {
            "model": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "learning_rate": LEARNING_RATE,
            "epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE
        },

        "lora_v1": {
            "rank": LORA_R,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "target_modules": [
                "query",
                "value"
            ]
        },

        "lora_improved": {
            "rank": IMPROVED_LORA_R,
            "alpha": IMPROVED_LORA_ALPHA,
            "dropout": IMPROVED_LORA_DROPOUT,
            "learning_rate": IMPROVED_LR,
            "target_modules": [
                "query",
                "key",
                "value",
                "dense"
            ]
        }
    }
}

with open(
    RESULTS_DIR / "experiment_config.json",
    "w"
) as f:

    json.dump(
        experiment_config,
        f,
        indent=4
    )

print("Experiment configuration saved.")